# note2quiz Week 3 data pipeline



In [1]:
# Cell 1: set up the workspace.
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.util
import json
import logging
import os
import platform
import re
import subprocess
import sys
from urllib.request import urlopen

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BASE_DIR = (Path('/content/week3_project') if IN_COLAB else
            Path(os.environ.get('NOTE2QUIZ_WORK_DIR', 'week3_project')).resolve())
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
PIPE_DIR = BASE_DIR / 'data' / 'pipeline_v1' / 'documents'
for folder in (RAW_DIR, PROCESSED_DIR, PIPE_DIR):
    folder.mkdir(parents=True, exist_ok=True)
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

if importlib.util.find_spec('pypdf') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pypdf==6.21.4'])
import pypdf
if pypdf.__version__ != '6.21.4':
    raise RuntimeError('This run uses pypdf 6.21.4. Install that version, then restart and run all cells.')
from pypdf import PdfReader
logging.getLogger('pypdf').setLevel(logging.ERROR)

ENVIRONMENT = 'Google Colab' if IN_COLAB else 'Local Python check'
print('Runtime:', ENVIRONMENT)
print('Python:', platform.python_version(), '| pypdf:', pypdf.__version__)
print('Raw folder:', RAW_DIR.relative_to(BASE_DIR).as_posix(), '| exists:', RAW_DIR.is_dir())
print('Processed folder:', PROCESSED_DIR.relative_to(BASE_DIR).as_posix(), '| exists:', PROCESSED_DIR.is_dir())

Runtime: Google Colab
Python: 3.13.18 | pypdf: 6.21.4
Raw folder: data/raw | exists: True
Processed folder: data/processed | exists: True


## Part A Choose and inspect the sources

The first sample uses computing and business notes. Think Python uses PDF pages 23 to 30, which cover Chapter 1. Teamwork in Business uses PDF pages 2 to 5, which include the opening example and team characteristics. These selected pages provide a small sample for testing the preparation method.

In [2]:
# Cell 2: source details and the selected pages.
SOURCE_COMMIT = '7e898e9011d0a898d46c173ffb0fbea056d8c85f'
REPOSITORY = 'https://github.com/adamiskandarr05/NOTE2QUIZ'
SOURCES = [
    {
        'source_id': 'S01',
        'filename': 'think_python_2e.pdf',
        'title': 'Think Python, second edition',
        'owner': 'Allen B. Downey / Green Tea Press',
        'version': 'Second edition, version 2.4.1, copyright 2015',
        'original_url': 'https://greenteapress.com/thinkpython2/thinkpython2.pdf',
        'licence': 'CC BY-NC 3.0',
        'licence_url': 'https://creativecommons.org/licenses/by-nc/3.0/',
        'licence_pdf_page': 4,
        'sha256': '6033000bac3f9268ad75ee041bf8e5d9a3bc8a33bb156399cbd014918cb59f94',
        'pages': [23, 30],
    },
    {
        'source_id': 'S02',
        'filename': 'teamwork_in_business.pdf',
        'title': 'Teamwork in Business, Fundamentals of Business Chapter 1',
        'owner': 'Ron Poff / Virginia Tech',
        'version': 'Fourth edition, 2023',
        'original_url': 'https://vtechworks.lib.vt.edu/bitstreams/38bd0a78-433e-4d0e-aa83-e0e0ff9b9b82/download',
        'licence': 'CC BY-NC-SA 4.0, except where otherwise noted',
        'licence_url': 'https://creativecommons.org/licenses/by-nc-sa/4.0/',
        'licence_pdf_page': 1,
        'sha256': 'a0a2d04aa27da310c90df785486c7f44cd7025a27e8f0899e6a866c66b0f07f6',
        'pages': [2, 5],
    },
]
CONFIG = {'method': 'paragraph-aware', 'chunk_size': 500,
          'page_aware': True, 'cleaning_version': 'week3-v1'}

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

def source_path(source):
    path = RAW_DIR / source['filename']
    if not path.is_file():
        raise FileNotFoundError('Missing source: ' + source['filename'])
    if path.suffix.lower() != '.pdf' or not 0 < path.stat().st_size <= 10 * 1024 * 1024:
        raise ValueError('Expected a non-empty PDF of at most 10 MB.')
    if file_hash(path) != source['sha256']:
        raise ValueError('Source checksum changed: ' + source['filename'])
    return path

In [3]:
# Cell 3: copy the original PDFs and verify the actual files.
for source in SOURCES:
    target = RAW_DIR / source['filename']
    url = (f'https://raw.githubusercontent.com/adamiskandarr05/NOTE2QUIZ/'
           f'{SOURCE_COMMIT}/data/{source["filename"]}')
    source['repository_url'] = url
    source['repository_path'] = 'data/' + source['filename']
    if not target.exists():
        with urlopen(url, timeout=60) as response:
            content = response.read(10 * 1024 * 1024 + 1)
        if len(content) > 10 * 1024 * 1024:
            raise ValueError('Downloaded file exceeds 10 MB.')
        if hashlib.sha256(content).hexdigest() != source['sha256']:
            raise ValueError('Downloaded file does not match the recorded source.')
        target.write_bytes(content)
    path = source_path(source)
    print(source['source_id'], path.name, '| bytes:', path.stat().st_size,
          '| SHA-256:', file_hash(path))
assert len(SOURCES) == 2
print('Verified 2 selected PDF files.')

S01 think_python_2e.pdf | bytes: 907689 | SHA-256: 6033000bac3f9268ad75ee041bf8e5d9a3bc8a33bb156399cbd014918cb59f94
S02 teamwork_in_business.pdf | bytes: 1424558 | SHA-256: a0a2d04aa27da310c90df785486c7f44cd7025a27e8f0899e6a866c66b0f07f6
Verified 2 selected PDF files.


In [4]:
# Cell 4: inspect real pages from both sources.
def read_sources():
    pages = []
    inspection = []
    for source in SOURCES:
        path = source_path(source)
        reader = PdfReader(path)
        count = len(reader.pages)
        first, last = source['pages']
        if count > 300 or not (1 <= first <= last <= count):
            raise ValueError('Page range or 300-page limit failed: ' + path.name)
        selected = []
        for number in range(first, last + 1):
            text = reader.pages[number - 1].extract_text() or ''
            if not text.strip():
                raise ValueError(f'No readable text in {path.name}, PDF page {number}.')
            item = {'source_id': source['source_id'], 'source': path.name,
                    'page': number, 'raw_text': text}
            pages.append(item)
            selected.append(item)
        inspection.append({
            'source_id': source['source_id'], 'filename': path.name,
            'bytes': path.stat().st_size, 'pdf_pages': count,
            'selected_pages': list(range(first, last + 1)),
            'selected_characters': sum(len(p['raw_text']) for p in selected),
            'empty_selected_pages': 0,
            'blank_line_breaks': sum(p['raw_text'].count('\n\n') for p in selected),
        })
    return pages, inspection

raw_pages, inspection = read_sources()
for item in inspection:
    print(json.dumps(item, ensure_ascii=False))
    sample = next(p for p in raw_pages if p['source_id'] == item['source_id'])
    print('PDF page', sample['page'], 'sample:', repr(sample['raw_text'][:300]))
print('Inspected selected pages:', len(raw_pages))

{"source_id": "S01", "filename": "think_python_2e.pdf", "bytes": 907689, "pdf_pages": 244, "selected_pages": [23, 24, 25, 26, 27, 28, 29, 30], "selected_characters": 16561, "empty_selected_pages": 0, "blank_line_breaks": 0}
PDF page 23 sample: 'Chapter 1\nThe way of the program\nThe goal of this book is to teach you to think like a computer scientist. This way of think-\ning combines some of the best features of mathematics, engineering, and natural science.\nLike mathematicians, computer scientists use formal languages to denote ideas (specif'
{"source_id": "S02", "filename": "teamwork_in_business.pdf", "bytes": 1424558, "pdf_pages": 20, "selected_pages": [2, 3, 4, 5], "selected_characters": 9680, "empty_selected_pages": 0, "blank_line_breaks": 0}
PDF page 2 sample: '1.  Teamwork in Business \nLearning Objectives \n• Define a team and describe its key characteristics. \n• Explain why organizations use teams and describe different types of teams. \n• Explain why teams may be effective 

## Part B Clean and compare the text

Only known page headers and footers are removed. Line endings and trailing spaces are cleaned. Internal line breaks and code indentation are kept. Broken words, joined words and reference numbers are left for later review because automatic repairs could change the meaning.

Both methods split each PDF page separately. The second method prefers a blank line, then a line break, then a word space near the 500-character limit. A long word can still need a fixed cut. The method uses text boundaries; it does not check the meaning of a passage.

In [5]:
# Cell 5: cleaning and the two splitting methods.
def clean_page(page):
    lines = page['raw_text'].replace('\r\n', '\n').replace('\r', '\n').replace('\f', '').split('\n')
    removed = 0
    if page['source_id'] == 'S01' and lines:
        printed = page['page'] - 22
        header = lines[0].strip()
        if (header == f'{printed} Chapter 1. The way of the program' or
            re.fullmatch(r'1\.\d+\. .+ ' + str(printed), header)):
            lines = lines[1:]
            removed += 1
    kept = []
    for line in lines:
        if (page['source_id'] == 'S02' and
            re.fullmatch(r'(?:Teamwork in Business\s*\|\s*\d+|\d+\s*\|\s*Teamwork in Business)', line.strip())):
            removed += 1
            continue
        kept.append(line.rstrip())
    text = '\n'.join(kept).strip('\n')
    if not text.strip():
        raise ValueError('Cleaning left an empty page.')
    return {**page, 'text': text, 'removed_lines': removed}

def cuts_word(text, end):
    if end >= len(text):
        return False
    direct = text[end - 1].isalnum() and text[end].isalnum()
    left, right = text[:end].rstrip(), text[end:].lstrip()
    hyphenated = bool(left.endswith('-') and right and right[0].isalpha())
    return direct or hyphenated

def chunk_spans(text, method, size):
    if size <= 0 or method not in ('fixed', 'paragraph-aware'):
        raise ValueError('Choose a supported method and positive chunk size.')
    if not text.strip():
        raise ValueError('Cannot prepare empty text.')
    spans = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        if method == 'paragraph-aware' and end < len(text):
            lower = start + size // 2
            found = None
            for separator in ('\n\n', '\n', ' ', '\t'):
                position = text.rfind(separator, lower, end)
                while position >= lower:
                    boundary = position + len(separator)
                    if not cuts_word(text, boundary):
                        found = boundary
                        break
                    position = text.rfind(separator, lower, position)
                if found is not None:
                    break
            if found is not None:
                end = found
        a, b = start, end
        while a < b and text[a] == '\n':
            a += 1
        while b > a and text[b - 1] == '\n':
            b -= 1
        if text[a:b].strip():
            spans.append((a, b))
        start = end
    return spans

def prepare_records(pages, configuration):
    if not pages:
        raise ValueError('No readable pages to prepare.')
    records = []
    for page in pages:
        source = next(s for s in SOURCES if s['source_id'] == page['source_id'])
        for index, (start, end) in enumerate(chunk_spans(
                page['text'], configuration['method'], configuration['chunk_size']), 1):
            records.append({
                'record_id': f'{page["source_id"]}-p{page["page"]:04d}-c{index:03d}',
                'source_id': page['source_id'], 'source': source['filename'],
                'page': page['page'], 'start_character': start, 'end_character': end,
                'text': page['text'][start:end], 'source_sha256': source['sha256'],
                'processing': configuration['method'] + '-page-aware-char-' + str(configuration['chunk_size']),
            })
    return records

cleaned_pages = [clean_page(p) for p in raw_pages]
for source in SOURCES:
    selected = [p for p in cleaned_pages if p['source_id'] == source['source_id']]
    print(source['source_id'], '| raw characters:', sum(len(p['raw_text']) for p in selected),
          '| cleaned characters:', sum(len(p['text']) for p in selected),
          '| header/footer lines removed:', sum(p['removed_lines'] for p in selected))

S01 | raw characters: 16561 | cleaned characters: 16339 | header/footer lines removed: 7
S02 | raw characters: 9680 | cleaned characters: 9452 | header/footer lines removed: 4


In [6]:
# Cell 6: compare actual counts and boundaries using the same pages.
page_lookup = {(p['source_id'], p['page']): p for p in cleaned_pages}
comparison = []
candidates = {}
for method in ('fixed', 'paragraph-aware'):
    configuration = {**CONFIG, 'method': method}
    rows = prepare_records(cleaned_pages, configuration)
    candidates[method] = rows
    cuts = []
    for row in rows:
        text = page_lookup[(row['source_id'], row['page'])]['text']
        end = row['end_character']
        if cuts_word(text, end):
            cuts.append(row['record_id'])
    item = {'method': method, 'records': len(rows),
            'minimum_characters': min(len(r['text']) for r in rows),
            'maximum_characters': max(len(r['text']) for r in rows),
            'cuts_inside_words': len(cuts),
            'records_by_source': {s['source_id']: sum(r['source_id'] == s['source_id'] for r in rows) for s in SOURCES}}
    comparison.append(item)
    print(json.dumps(item))
    print('Example end:', repr(rows[0]['text'][-100:]))
    print('Next start:', repr(rows[1]['text'][:100]))
assert comparison[1]['cuts_inside_words'] <= comparison[0]['cuts_inside_words']
print('Selected:', CONFIG['method'], '| fewer cuts inside words; may create more pieces.')

{"method": "fixed", "records": 57, "minimum_characters": 63, "maximum_characters": 500, "cuts_inside_words": 28, "records_by_source": {"S01": 37, "S02": 20}}
Example end: ' evaluating tradeoffs among alternatives. Like scientists, they observe the behav-\nior of complex sy'
Next start: 'stems, form hypotheses, and test predictions.\nThe single most important skill for a computer scienti'
{"method": "paragraph-aware", "records": 62, "minimum_characters": 60, "maximum_characters": 499, "cuts_inside_words": 0, "records_by_source": {"S01": 39, "S02": 23}}
Example end: 'nd evaluating tradeoffs among alternatives. Like scientists, they observe the behav-\nior of complex '
Next start: 'systems, form hypotheses, and test predictions.\nThe single most important skill for a computer scien'
Selected: paragraph-aware | fewer cuts inside words; may create more pieces.


In [7]:
# Cell 7: save and check the chosen Part B output.
def validate_records(rows, pages, configuration):
    if not rows or len({r['record_id'] for r in rows}) != len(rows):
        raise ValueError('Missing records or repeated record IDs.')
    lookup = {(p['source_id'], p['page']): p for p in pages}
    for row in rows:
        page = lookup[(row['source_id'], row['page'])]
        source = next(s for s in SOURCES if s['source_id'] == row['source_id'])
        start, end = row['start_character'], row['end_character']
        assert isinstance(row['page'], int) and source['pages'][0] <= row['page'] <= source['pages'][1]
        assert 0 <= start < end <= len(page['text'])
        assert 0 < len(row['text']) <= configuration['chunk_size'] and row['text'].strip()
        assert row['text'] == page['text'][start:end]
        assert row['source'] == source['filename'] and row['source_sha256'] == source['sha256']
    for key, page in lookup.items():
        joined = ''.join(r['text'] for r in rows if (r['source_id'], r['page']) == key)
        assert re.sub(r'\s', '', joined) == re.sub(r'\s', '', page['text']), 'Page content was lost.'

def write_records(path, rows):
    with Path(path).open('w', encoding='utf-8', newline='\n') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + '\n')
    if not Path(path).is_file() or Path(path).stat().st_size == 0:
        raise ValueError('Prepared output missing or empty.')
    exported = [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]
    assert exported == rows, 'Saved records do not match.'

records = prepare_records(cleaned_pages, CONFIG)
validate_records(records, cleaned_pages, CONFIG)
prepared_path = PROCESSED_DIR / 'prepared_text.jsonl'
write_records(prepared_path, records)
print('Part B validation: PASS')
print('Output: data/processed/prepared_text.jsonl')
print('Records:', len(records), '| bytes:', prepared_path.stat().st_size)
for source in SOURCES:
    row = next(r for r in records if r['source_id'] == source['source_id'])
    print(row['record_id'], '| file:', row['source'], '| PDF page:', row['page'], '| text:', repr(row['text'][:180]))

Part B validation: PASS
Output: data/processed/prepared_text.jsonl
Records: 62 | bytes: 44580
S01-p0023-c001 | file: think_python_2e.pdf | PDF page: 23 | text: 'Chapter 1\nThe way of the program\nThe goal of this book is to teach you to think like a computer scientist. This way of think-\ning combines some of the best features of mathematics,'
S02-p0002-c001 | file: teamwork_in_business.pdf | PDF page: 2 | text: '1.  Teamwork in Business\nLearning Objectives\n• Define a team and describe its key characteristics.\n• Explain why organizations use teams and describe different types of teams.\n• Ex'


In [8]:
# Cell 8: save and verify Part B lineage.
lineage_records = []
for source in SOURCES:
    lineage_records.append({
        'source_id': source['source_id'], 'source_filename': source['filename'],
        'repository_path': source['repository_path'], 'repository_url': source['repository_url'],
        'original_url': source['original_url'], 'source_sha256': file_hash(source_path(source)),
        'source_version': source['version'], 'owner': source['owner'],
        'access_rights': source['licence'], 'licence_url': source['licence_url'],
        'selected_pdf_pages': source['pages'],
        'prepared_output': 'data/processed/prepared_text.jsonl',
        'prepared_sha256': file_hash(prepared_path),
        'prepared_records': sum(r['source_id'] == source['source_id'] for r in records),
        'processing_note': 'Known page headers/footers and trailing spaces removed; page-aware paragraph/line/word splitting at 500 characters.',
        'status': 'Prepared; question quality not evaluated',
    })
manifest = {
    'pipeline_version': 'week3-v1', 'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'runtime': ENVIRONMENT, 'python': platform.python_version(), 'pypdf': pypdf.__version__,
    'configuration': CONFIG, 'requirements': ['FR1', 'FR3', 'RAI1'],
    'records': lineage_records,
    'limits': ['PDF position is retained; printed page numbers can differ.',
               'Text extraction can lose layout, spacing and figures.',
               'Selected pages only; no claim of complete course coverage.'],
}
manifest_path = BASE_DIR / 'data' / 'week3_manifest.json'
save_json(manifest_path, manifest)
saved = json.loads(manifest_path.read_text(encoding='utf-8'))
assert len(saved['records']) == 2
for row in saved['records']:
    assert file_hash(RAW_DIR / row['source_filename']) == row['source_sha256']
    assert file_hash(BASE_DIR / row['prepared_output']) == row['prepared_sha256']
    assert row['prepared_records'] > 0
print('COMMON LINEAGE VALIDATION: PASS')
print('Verified source-to-output links:', len(saved['records']))

COMMON LINEAGE VALIDATION: PASS
Verified source-to-output links: 2


## Part C Run the same steps as a pipeline

The function below reads the selected files again, prepares the passages using the same settings and checks the saved output. It records source details, checksums and output files.

In [9]:
# Cell 9: executable V1 document pipeline, matching Part B.
def run_pipeline():
    pages, details = read_sources()
    cleaned = [clean_page(p) for p in pages]
    rows = prepare_records(cleaned, CONFIG)
    validate_records(rows, cleaned, CONFIG)
    output = PIPE_DIR / 'prepared_text.jsonl'
    write_records(output, rows)
    sources = [
        {'source_id': s['source_id'], 'filename': s['filename'],
         'repository_url': s['repository_url'], 'original_url': s['original_url'],
         'source_version': s['version'], 'access_rights': s['licence'],
         'licence_url': s['licence_url'], 'sha256': file_hash(source_path(s)),
         'selected_pdf_pages': s['pages']}
        for s in SOURCES
    ]
    mappings = [
        {'source_id': s['source_id'], 'source': s['filename'],
         'output': 'data/pipeline_v1/documents/prepared_text.jsonl',
         'records': sum(r['source_id'] == s['source_id'] for r in rows)}
        for s in SOURCES
    ]
    validation = {
        'output_count': 1, 'output_sha256': file_hash(output),
        'output_bytes': output.stat().st_size, 'record_count': len(rows),
        'unique_ids': True, 'source_and_page_checks': 'PASS',
        'saved_records_match': True, 'part_b_output_matches': file_hash(output) == file_hash(prepared_path),
    }
    assert validation['part_b_output_matches'], 'Part B and Part C outputs differ.'
    record = {
        'pipeline_version': 'week3-v1', 'generated_at_utc': datetime.now(timezone.utc).isoformat(),
        'runtime': ENVIRONMENT, 'python': platform.python_version(), 'pypdf': pypdf.__version__,
        'data_type': 'documents', 'requirements': ['FR1', 'FR3', 'RAI1'],
        'configuration': CONFIG, 'sources': sources, 'inspection': details,
        'outputs': ['data/pipeline_v1/documents/prepared_text.jsonl'],
        'source_output_mapping': mappings, 'validation': validation,
    }
    save_json(PIPE_DIR / 'pipeline_manifest.json', record)
    print('SUCCESS: documents | Inputs:', len(sources), '| Outputs:', validation['output_count'])
    print('Records:', len(rows), '| SHA-256:', validation['output_sha256'])
    return record

first_run = run_pipeline()

SUCCESS: documents | Inputs: 2 | Outputs: 1
Records: 62 | SHA-256: 2f111a9a2c345d260583069711a677a06b88d9cdc4a1ea1859eac173675bed3f


In [10]:
# Cell 10: rerun and check the results.
second_run = run_pipeline()
assert first_run['sources'] == second_run['sources']
assert first_run['validation'] == second_run['validation']
assert first_run['source_output_mapping'] == second_run['source_output_mapping']
second_run['repeatability'] = {
    'status': 'PASS', 'runs': 2, 'sources_match': True,
    'prepared_outputs_match': True, 'source_output_mappings_match': True,
    'note': 'Run timestamps may differ. This is a same-environment repeatability check.'
}
save_json(PIPE_DIR / 'pipeline_manifest.json', second_run)
print('PASS: source checksums, prepared outputs, sizes, counts and mappings match.')
print('PASS: Part B and Part C prepared text match.')

SUCCESS: documents | Inputs: 2 | Outputs: 1
Records: 62 | SHA-256: 2f111a9a2c345d260583069711a677a06b88d9cdc4a1ea1859eac173675bed3f
PASS: source checksums, prepared outputs, sizes, counts and mappings match.
PASS: Part B and Part C prepared text match.


In [11]:
# Cell 11: show the response to two controlled failures.
failure_checks = []
try:
    source_path({**SOURCES[0], 'filename': '__missing_for_check__.pdf'})
except FileNotFoundError:
    failure_checks.append({'case': 'Missing source file', 'response': 'Stopped before reading', 'status': 'PASS'})
else:
    raise AssertionError('Missing source was not blocked.')
try:
    chunk_spans('', CONFIG['method'], CONFIG['chunk_size'])
except ValueError:
    failure_checks.append({'case': 'No readable text', 'response': 'Stopped before preparing empty passages', 'status': 'PASS'})
else:
    raise AssertionError('Empty text was not blocked.')
for item in failure_checks:
    print(item['status'] + ': ' + item['case'] + ' -> ' + item['response'])
second_run['failure_checks'] = failure_checks
save_json(PIPE_DIR / 'pipeline_manifest.json', second_run)

PASS: Missing source file -> Stopped before reading
PASS: No readable text -> Stopped before preparing empty passages
